# 803. Bricks Falling When Hit

## Topic Alignment
- **Role Relevance**: Model structural stability analysis where removing support elements causes cascading failures.
- **Scenario**: Analyze infrastructure dependencies, network resilience, or physical structure integrity where component removal affects connected systems.

## Metadata Summary
- Source: [Bricks Falling When Hit](https://leetcode.com/problems/bricks-falling-when-hit/)
- Tags: `Union-Find`, `Array`, `Matrix`, `Reverse Thinking`
- Difficulty: Hard
- Recommended Priority: High

## Problem Statement
You are given an `m x n` binary grid, where each `1` represents a brick and `0` represents an empty space. A brick is stable if:
- It is directly connected to the top of the grid, or
- At least one other brick in its four adjacent cells is stable.

You are also given an array `hits`, which is a sequence of erasures we want to apply. Each time we want to erase the brick at the location `hits[i] = (rowi, coli)`. The brick on that location (if it exists) will disappear. Some other bricks may no longer be stable because of that erasure and will fall.

Return an array `result`, where each `result[i]` is the number of bricks that will fall after the `i`th erasure is applied.

Note that an erasure may refer to a location with no brick, and if it does, no bricks drop.

## Progressive Hints
- Hint 1: Direct simulation is hard. Consider processing hits in reverse order.
- Hint 2: Instead of removing bricks, add them back and see how many become stable.
- Hint 3: Use Union-Find with a virtual "roof" node connected to the top row.
- Hint 4: Track the size of the component connected to the roof before and after adding each brick back.

## Solution Overview
Use reverse thinking: start with all bricks removed, then add them back in reverse order. Track how many bricks become connected to the roof when each brick is restored using Union-Find.

## Detailed Explanation
1. **Initial state**: Mark all hit locations in the grid and create the final state after all hits.
2. **Virtual roof**: Add a virtual node (index m*n) connected to all bricks in the top row.
3. **Build final state**: Union all remaining adjacent bricks with Union-Find.
4. **Reverse process**: Add hits back in reverse order:
   - Before adding: count bricks connected to roof.
   - Add the brick back and union with stable neighbors.
   - After adding: count bricks connected to roof.
   - Difference = bricks that fell due to this hit (minus 1 for the brick itself).
5. **Edge case**: If the hit location had no brick initially, result is 0.
6. **Return results in original order**: Reverse the computed results.

## Complexity Trade-off Table
| Approach | Time Complexity | Space Complexity | Notes |
| --- | --- | --- | --- |
| Reverse Union-Find | O(m*n + k) α(m*n) | O(m*n) | Optimal for this problem |
| Direct simulation | O(k * m*n) | O(m*n) | Recompute stability after each hit |
| DFS per hit | O(k * m*n) | O(m*n) | Inefficient for large k |

## Reference Implementation

In [ ]:
from typing import List


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.size = [1] * n
    
    def find(self, x: int) -> int:
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]
    
    def union(self, x: int, y: int):
        root_x, root_y = self.find(x), self.find(y)
        if root_x != root_y:
            self.parent[root_y] = root_x
            self.size[root_x] += self.size[root_y]
    
    def get_size(self, x: int) -> int:
        return self.size[self.find(x)]


def hitBricks(grid: List[List[int]], hits: List[List[int]]) -> List[int]:
    m, n = len(grid), len(grid[0])
    
    # Copy grid and mark all hits
    grid_copy = [row[:] for row in grid]
    for r, c in hits:
        grid_copy[r][c] = 0
    
    # Union-Find with virtual roof node at index m*n
    uf = UnionFind(m * n + 1)
    roof = m * n
    
    def get_index(r: int, c: int) -> int:
        return r * n + c
    
    # Build initial stable structure
    for i in range(m):
        for j in range(n):
            if grid_copy[i][j] == 1:
                # Connect to roof if in top row
                if i == 0:
                    uf.union(get_index(i, j), roof)
                # Connect to adjacent bricks
                if i > 0 and grid_copy[i-1][j] == 1:
                    uf.union(get_index(i, j), get_index(i-1, j))
                if j > 0 and grid_copy[i][j-1] == 1:
                    uf.union(get_index(i, j), get_index(i, j-1))
    
    # Process hits in reverse
    result = []
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    for r, c in reversed(hits):
        # If no brick originally, no bricks fall
        if grid[r][c] == 0:
            result.append(0)
            continue
        
        # Count stable bricks before adding this brick back
        prev_roof_size = uf.get_size(roof)
        
        # Add brick back
        grid_copy[r][c] = 1
        
        # Connect to roof if in top row
        if r == 0:
            uf.union(get_index(r, c), roof)
        
        # Connect to adjacent stable bricks
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and grid_copy[nr][nc] == 1:
                uf.union(get_index(r, c), get_index(nr, nc))
        
        # Count stable bricks after adding this brick back
        curr_roof_size = uf.get_size(roof)
        
        # Bricks that fell = increase in roof size - 1 (the brick itself)
        fallen = max(0, curr_roof_size - prev_roof_size - 1)
        result.append(fallen)
    
    return result[::-1]

## Validation

In [ ]:
assert hitBricks([[1,0,0,0],[1,1,1,0]], [[1,0]]) == [2]
assert hitBricks([[1,0,0,0],[1,1,0,0]], [[1,1],[1,0]]) == [0,0]
assert hitBricks([[1],[1],[1],[1],[1]], [[3,0],[4,0],[1,0],[2,0],[0,0]]) == [1,0,1,0,0]
assert hitBricks([[1,0,1],[1,1,1]], [[0,0],[0,2],[1,1]]) == [0,3,0]
print('All tests passed for LC 803.')

## Complexity Analysis
- Time Complexity: O((m*n + k) α(m*n)), where m*n is grid size, k is number of hits, and α is inverse Ackermann function.
- Space Complexity: O(m*n) for Union-Find structure and grid copy.
- Bottleneck: Initial grid processing and reverse hit processing with Union-Find operations.

## Edge Cases & Pitfalls
- Hit on empty cell: Should return 0 (no brick to remove).
- Hit on isolated brick: May fall itself but not cause others to fall.
- Top row hits: Critical as they directly connect to roof.
- Virtual roof: Essential for tracking stability efficiently.

## Follow-up Variants
- Handle brick addition queries instead of removal.
- Find minimum hits to make all bricks unstable.
- Extend to 3D structures with gravity.

## Takeaways
- Reverse thinking transforms hard deletion problems into easier addition problems.
- Virtual nodes (like roof) simplify connectivity tracking.
- Union-Find with size tracking enables efficient component analysis.
- Processing operations in reverse order is a powerful technique for cascading effect problems.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| 305 | Number of Islands II | Union-Find with dynamic additions |
| 1970 | Last Day Where You Can Still Cross | Reverse time Union-Find |
| 1568 | Minimum Number of Days to Disconnect Island | Graph connectivity analysis |